# 01 · Data sources: from a public API to `Evidence`

**What you will learn**
1. The `Evidence` model: the one shape every source returns
2. The shared `HttpClient` (retries + cache) and the ACID `Store` behind the cache
3. Each of the four sources, called directly
4. Why the cache matters (speed, fewer calls, same answer)

```
openFDA labels ─┐
openFDA FAERS ──┤   source.search(...)  ──►  list[Evidence]  ──►  tools (notebook 02)  ──►  agents (notebook 03)
ClinicalTrials ─┤
PubMed ─────────┘
```

In [1]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

## 1. Config
Everything is built from `configs/default.yaml`. The config is a plain dict, so you can tweak it here.
Using `store.path = ":memory:"` keeps this notebook's data out of the real run log.

In [2]:
from pharma_agent.config import load_config
cfg = load_config(overrides={"store.path": ":memory:"})
print("Sources to load:", cfg["sources"])
print("LLM roles:", cfg["llm"]["roles"])

Sources to load: ['openfda_labels', 'openfda_events', 'clinicaltrials', 'pubmed']
LLM roles: {'supervisor': 'openai/gpt-oss-120b', 'specialist': 'openai/gpt-oss-20b', 'synthesizer': 'openai/gpt-oss-120b'}


## 2. `Evidence`: the contract between sources and agents
Every source returns a list of these. The `id` is what the final answer cites, and the guardrail later checks
that every cited id really came from a tool.

In [3]:
from pharma_agent.schemas import Evidence
print(list(Evidence.model_json_schema()["properties"]))

['id', 'source', 'title', 'text', 'url', 'metadata']


## 3. The HTTP client and the ACID store
`build_http` gives one client shared by all sources (dependency injection). It retries on 429/5xx with
exponential back-off and caches answers in SQLite. Each cache write is a transaction, so a crash never
leaves a half-written row.

In [4]:
from pharma_agent.wiring import build_http, build_store
from pharma_agent.sources import build_sources

store = build_store(cfg)            # SQLite (in memory here)
http = build_http(cfg, store)       # retries + cache
sources = {s.name: s for s in build_sources(cfg["sources"], http)}
for name, s in sources.items():
    print(f"{name:26s} <- {type(s).__name__}")

search_drug_labels         <- OpenFDALabels
get_adverse_event_counts   <- OpenFDAAdverseEvents
search_clinical_trials     <- ClinicalTrials
search_pubmed              <- PubMed


## 4. Each source, called directly
### FDA drug label: one section (here, the boxed warning and warnings)

In [5]:
[label] = sources["search_drug_labels"].search(drug="metformin", section="warnings")
print(label.id)
print(label.title)
print(label.url)
print(label.metadata)
print(label.text[:500])

fda-label:0230a3c3:warnings
FDA label: Metformin Hydrochloride (METFORMIN HYDROCHLORIDE), warnings
https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid=0230a3c3-34cf-478b-88c3-4d96e5300018
{'label_fields': ['boxed_warning', 'warnings_and_cautions'], 'has_boxed_warning': True, 'effective_time': '20261109', 'manufacturer': 'PD-Rx Pharmaceuticals, Inc.'}
[BOXED_WARNING] WARNING: LACTIC ACIDOSIS Postmarketing cases of metformin-associated lactic acidosis have resulted in death, hypothermia, hypotension, and resistant bradyarrhythmias. The onset of metformin-associated lactic acidosis is often subtle, accompanied only by nonspecific symptoms such as malaise, myalgias, respiratory distress, somnolence, and abdominal pain. Metformin-associated lactic acidosis was characterized by elevated blood lactate levels (> 5 mmol/Liter), anion gap acidosis (wi


### FAERS: most reported adverse reactions (reports, not proof of causation)

In [6]:
[faers] = sources["get_adverse_event_counts"].search(drug="atorvastatin", top_n=8)
print(faers.text)

Most reported adverse reactions (FAERS, spontaneous reports, not causal): Fatigue: 14,618 reports; Drug ineffective: 13,513 reports; Nausea: 13,086 reports; Diarrhoea: 11,799 reports; Dyspnoea: 11,797 reports; Type 2 diabetes mellitus: 11,285 reports; Pain: 10,551 reports; Headache: 10,427 reports


### ClinicalTrials.gov: filter by drug, condition and status

In [7]:
trials = sources["search_clinical_trials"].search(condition="obesity", intervention="semaglutide", status="RECRUITING", limit=3)
for t in trials:
    print(t.id, "|", t.title[:90])
    print("   ", t.text[:200], "\n")

nct:NCT06374875 | NCT06374875: Fibrosis Lessens After Metabolic Surgery
    Status: RECRUITING; Phase: PHASE4; Conditions: Metabolic Dysfunction-associated Steatotic Liver Disease (MASLD), Non-Alcoholic Fatty Liver Disease, Metabolic Dysfunction-Associated Steatohepatitis (MA 

nct:NCT07732426 | NCT07732426: Effects of Semaglutide on Brain Dopamine Responses to Food Cues in Adults Wit
    Status: RECRUITING; Phase: NA; Conditions: Food Addiction Tendency, Obesity, Overweight; Interventions: Semaglutide; Enrollment: 30; Sponsor: Seoul National University Hospital; Start: 2026-07-15. Sum 

nct:NCT07901777 | NCT07901777: Two-Year Weight Loss With GLP-1 Medications Compared With Sleeve Gastrectomy 
    Status: RECRUITING; Phase: n/a; Conditions: Obesity & Overweight; Interventions: GLP-1 Receptor Agonist Therapy, Laparoscopic Sleeve Gastrectomy; Enrollment: 200; Sponsor: Yuzuncu Yil University; Star 



### PubMed: search, then fetch abstracts

In [8]:
papers = sources["search_pubmed"].search(query="SGLT2 inhibitors heart failure outcomes", limit=3)
for p in papers:
    print(p.id, "|", p.title[:100])

pmid:39576469 | SGLT2 Inhibitors in Patients with Heart Failure: A Model-Based Meta-Analysis. (Clinical pharmacokine
pmid:40996542 | Use of SGLT2 inhibitors in pediatric heart failure: a multi-center study. (Pediatric cardiology, 202
pmid:38806171 | 2024 update in heart failure. (ESC heart failure, 2025)


### Nothing found is a normal answer, not an error

In [9]:
print(sources["search_drug_labels"].search(drug="notarealdrugxyz"))

[]


## 5. The cache in action
The second identical request is served from SQLite: same answer, no network call.

In [10]:
import time
url, params = "https://api.fda.gov/drug/label.json", {"search": 'openfda.generic_name:"warfarin"', "limit": 1}
for attempt in (1, 2):
    t0 = time.perf_counter()
    res = http.get(url, params)
    print(f"call {attempt}: status={res.status} from_cache={res.from_cache} {1000 * (time.perf_counter() - t0):.0f} ms")
print("rows in http_cache:", store.count("http_cache"))

call 1: status=200 from_cache=False 374 ms
call 2: status=200 from_cache=True 0 ms
rows in http_cache: 7


## Try it yourself
* Change `section=` to `"drug_interactions"`, `"pregnancy"` or `"mechanism"`.
* Search trials with `status="COMPLETED"`.
* Next: **02_tools_and_mcp.ipynb** turns these sources into tools.